# Research: fixing VCC 2026 MSE

Scope: MSE research; all six metrics are safeguards for cell-based choices. No leaderboard uploads. Existing jobs and pre-existing edits are preserved. Code lives in this notebook; input snapshots, derived arrays, tables, and PNGs live in ignored `lib/research_mse/` and `results/research_mse/`. Each completed phase is committed separately.

Assumptions: use the pinned `cell-eval2==0.18.0` implementation and the existing 75-target H1 reference; H1 is calibration, not an untouched test. Never train on H1 for H1 predictions. Use final models only for A–C. For E1a, “full controls” means the full **generator half** on H1 (preserving the disjoint scoring half), and all available challenge controls on A–C. All improvements must be judged using the scorer's own bounded correction, not an uncapped surrogate.

## Phase 1 — exact definition and reproduction


In [ ]:
import os, sys, json, hashlib, zipfile, argparse, copy, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import anndata as ad
import scipy.sparse as sp
import importlib.metadata as metadata
import cell_eval2
V = Path.cwd()
SRC = V / 'lib/research_mse/source'
OUT = V / 'results/research_mse'
OUT.mkdir(parents=True, exist_ok=True)
REF = V / 'lib/vcc_2026_3/val_final'
RUN = V / 'lib/vcc_2026_4/runs/dev_nopie'
PKG = Path(cell_eval2.__file__).parent
sys.path.insert(0, str(V / 'code/vcc_2026_3c'))
import vcc3_generate as gen
from vcc2_common import seed_for
from cell_eval2.metrics import delta as metric_delta
from cell_eval2.moments import GroupMoments, jackknife_correction
from cell_eval2.score import _replicate_entries
from cell_eval2.scoring import score_one
assert metadata.version('cell-eval2') == '0.18.0'
wheel = Path('/tmp/research_mse_scorer/cell_eval2-0.18.0-py3-none-any.whl')
if wheel.exists():
    with zipfile.ZipFile(wheel) as z:
        mismatches = [p for p in z.namelist() if p.startswith('cell_eval2/') and p.endswith(('.py','.yaml')) and z.read(p) != (PKG.parent/p).read_bytes()]
    assert not mismatches, mismatches
    print('Installed scorer exactly matches the downloaded 0.18.0 wheel.')
files=['metrics/delta.py','moments.py','catalog.py','run.py','baseline.py','anchor.py','scoring.py','score.py','configs/vcc2026.yaml']
provenance={f:hashlib.sha256((PKG/f).read_bytes()).hexdigest() for f in files}
provenance['generator_at_start']=hashlib.sha256(Path(gen.__file__).read_bytes()).hexdigest()
(OUT/'source_hashes.json').write_text(json.dumps(provenance,indent=2))
print('Scorer:', PKG)


Let $y_{ig}$ be integer raw counts, $P_{kg}=\sum_i y_{ikg}$, and $\mu_{kg}=\log(1+50000 P_{kg}/\sum_gP_{kg})$. The log is natural. All panel genes enter library normalization. Let $G_k$ be the panel excluding **only knockdown k's own target gene**, and $w_k=1/|G_k|$.

The delete-one-cell jackknife is $J_k=(n_k-1)/n_k\sum_g\sum_i(v_{ikg}-\bar v_{kg})^2$, where $v_{ikg}=\log(1+50000(P_{kg}-y_{ikg})/(\sum_gP_{kg}-\sum_gy_{ikg}))$. It estimates sampling variance of the log-normalized group sum, not variance of a mean of cell-level logs.

For predicted cells, the first bound is $q_k=\min(J_k^{pred},J_k^{real})$. Define claim $C=\sum_kw_kq_k$ and budget $B=\sum_g\sum_{k:g\in G_k}w_k(\mu^{pred}_{kg}-\bar\mu^{pred}_{g,w})^2$, using the weighted mean over rows retaining gene g. The second bound is $r=\min(1,B/C)$ (no correction when claim is zero). The accepted prediction correction is $r q_k$.

The per-knockdown numerator is $N_k=w_k[\sum_{g\in G_k}(\mu^{pred}_{kg}-\mu^{real}_{kg})^2-J_k^{real}-rq_k]$. The uncapped diagnostic instead subtracts the full $J_k^{pred}$. The denominator is $D_k=w_k[\sum_{g\in G_k}(\mu^{real}_{kg}-\mu^{ctrl}_{g})^2-J_k^{real}-J^{ctrl}]$. **Corrections stay summed over all genes even though the own-target squared residual is removed.** There is no unique scored per-gene correction allocation in these scalar caches; per-gene diagnostics must report this explicitly.

Raw MSE is **$U=\sum_kN_k/\sum_kD_k$**, using paired finite rows. Individual numerators and denominators can be negative; they are not clipped. The summed denominator must be positive. “Capped” caps the noise correction, not errors, genes, targets, or the raw score. The real-control >5 CPM gate is for DE; it does not remove genes from expression MSE. Other panel targets remain in MSE (PDS removes all panel targets).

Scaled MSE is $\operatorname{clip}((b-U)/(b-a),0,1)$, where b is the measured generic baseline and a the split-half replicate anchor. Report the linear expression before clipping separately. Overall averages the six scaled members; MSE alone has this [0,1] clamp.


In [ ]:
# Executable, version-pinned source quotations with file and line numbers.
quotes = {
 'configs/vcc2026.yaml': [(10,26),(60,64)],
 'metrics/delta.py': [(41,43),(213,227),(280,290),(697,700),(832,835),(854,864),(878,881),(1228,1236)],
 'moments.py': [(173,180)], 'catalog.py':[(345,357)], 'run.py':[(1896,1905),(1912,1913)],
 'baseline.py':[(112,125),(167,181),(233,239),(323,327),(333,335)],
 'anchor.py':[(117,121),(135,140),(178,195),(277,287)],
 'score.py':[(394,401)]}
for file, spans in quotes.items():
    lines=(PKG/file).read_text().splitlines()
    text='\n'.join(f'{i}: {lines[i-1]}' for a,b in spans for i in range(a,min(b,len(lines))+1))
    display(Markdown(f'**`cell_eval2/{file}`**\n```python\n{text}\n```'))


The generic baseline first takes **each real non-control perturbation's mean raw counts**, then averages those means equally over perturbations. For gene g it omits the perturbation targeting g; it includes the scored perturbation for other genes. This is not a leave-one-knockdown-out baseline and is not the arithmetic mean of log-normalized pseudobulks. It rescales resampled real controls gene-wise by generic mean / control mean, with float32 fractional counts (an explicit baseline exception to submission integer requirements). Therefore its emitted pseudobulk contains sampling noise and need not equal the noise-free log of its expected counts. Genes absent in controls cannot be induced by that multiplicative construction.

The replicate anchor uses five independent seeded disjoint half-splits within each perturbation and controls, scoring half B against half A, with each side's own controls. The anchor is the mean of **five aggregate scores**, not pooled per-target values and not a Spearman–Brown-corrected MSE. The H1 frozen bundle is the authority for its exact seeds and settings.


In [ ]:
genes=pd.read_csv(V/'data/gene_names.csv')['gene_name'].astype(str).to_numpy()
G=len(genes); gp={g:i for i,g in enumerate(genes)}
z=np.load(next((REF/'real_cache').glob('*.moments.npz')))
print('Reference cache fields:', z.files)
perts=z['perts'].astype(str); means=z['means'].astype(float); jk=z['jk'].astype(float)
targets=sorted(set(perts)-{'non-targeting'}); T=len(targets); rows=np.arange(T)
ridx={p:i for i,p in enumerate(perts)}; ri=np.array([ridx[t] for t in targets]); ci=ridx['non-targeting']
R=means[ri]; ctrl=means[ci]; Jr=jk[ri]; Jc=jk[ci]
own=np.array([gp.get(t,-1) for t in targets]); nrow=np.where(own>=0,G-1,G); weights=1/nrow
mask=np.ones((T,G),bool); mask[rows[own>=0],own[own>=0]]=False
Dden=(((R-ctrl)**2*mask).sum(1)-Jr-Jc)/nrow
base=pd.read_csv(REF/'bundle/baseline_agg.csv').set_index('statistic').loc['mean'].to_dict()
anchor=pd.read_parquet(REF/'bundle/anchor_agg.parquet')
entries=_replicate_entries(base,anchor); MSE='expr_mse_unbiased_capped_norm'
b=float(base[MSE]); a=float(anchor.set_index('metric').loc[MSE,'replicate'])
def scaled_mse(raw): return (b-raw)/(b-a)
def tidy(path):
    df=pd.read_csv(path); df.columns=['perturbation','metric','value']
    return df.pivot(index='perturbation',columns='metric',values='value').loc[targets]
bl=tidy(REF/'baseline/baseline_results.csv')
np.testing.assert_allclose(Dden,bl.expr_distance_unbiased,rtol=2e-12,atol=1e-14)
print(f'{T} targets × {G} genes; denominator sum={Dden.sum():.12g}; baseline={b:.12g}; replicate={a:.12g}')
manifest=json.loads((REF/'bundle/manifest.json').read_text())
print('Replicate seeds:',manifest['derived_seeds'])
arm_paths={
 'arm1':SRC/'h1_bench_v5a1_p1/a1_arm1_s1_floor',
 'arm2':SRC/'h1_bench_v5a1_p1/a1_arm2_s1',
 'arm7':SRC/'h1_bench_v5a1_p3/a1_arm7_s2_floor'}
phase1_rows=[]; phase1_summary=[]
for name,path in arm_paths.items():
    df=tidy(path/'results.csv')
    np.testing.assert_allclose(df.expr_distance_unbiased,Dden,rtol=2e-12,atol=1e-14)
    raw=df.expr_mse_unbiased_capped.sum()/Dden.sum()
    reported=pd.read_csv(path/'agg_results.csv').set_index('statistic').loc['mean',MSE]
    np.testing.assert_allclose(raw,reported,rtol=2e-12)
    part=pd.DataFrame({'arm':name,'target':targets,'normalizer':Dden,
       'uncapped_error':df.expr_mse_unbiased.to_numpy(),'capped_error':df.expr_mse_unbiased_capped.to_numpy()})
    part['correction_withheld']=part.capped_error-part.uncapped_error
    phase1_rows.append(part)
    phase1_summary.append({'arm':name,'raw_capped':raw,'raw_uncapped':df.expr_mse_unbiased.sum()/Dden.sum(),
                          'mse_unclamped':scaled_mse(raw),'mse_scaled':np.clip(scaled_mse(raw),0,1),
                          'row_reaggregation_difference':raw-reported})
phase1_rows=pd.concat(phase1_rows,ignore_index=True)
phase1_rows.to_csv(OUT/'phase1_per_target.csv',index=False)
display(pd.DataFrame(phase1_summary))
display(phase1_rows)
print('This table independently reconstructs denominators and aggregate ratios from the saved scorer per-target numerators. Phase 2 also regenerates cells and independently rebuilds those numerators.')


In [ ]:
# Exact metric audit implementation, cross-checked against official functions below.
excl={i:int(j) for i,j in enumerate(own) if j>=0}
real_moments=GroupMoments(perts,np.asarray(z['counts']),np.asarray(z['sumsq']),jk)
def exact_mse(P,Jpred=None,verify=False):
    P=np.asarray(P,float); Jpred=np.zeros(T) if Jpred is None else np.asarray(Jpred,float)
    plugin=((P-R)**2*mask).sum(1)/nrow
    claim_rows=np.minimum(Jpred,Jr)
    claim=float((claim_rows/nrow).sum())
    budget=metric_delta._across_pert_budget(P,excl,weights)
    ratio=min(1.0,budget/claim) if claim>0 else 1.0
    uncapped=plugin-(Jr+Jpred)/nrow
    capped=plugin-(Jr+ratio*claim_rows)/nrow
    if verify:
        pm=GroupMoments(np.array(targets),np.full(T,400),np.zeros(T),Jpred)
        kw=dict(pred_bulk=(np.array(targets),P),real_bulk=(perts,means),pred_moments=pm,
                real_moments=real_moments,control='non-targeting',genes=genes,comparator='bulk_lognorm')
        official=metric_delta.mse_unbiased_capped(**kw)
        np.testing.assert_allclose(capped,[official[t] for t in targets],rtol=1e-10,atol=1e-13)
    return dict(raw=capped.sum()/Dden.sum(),raw_uncapped=uncapped.sum()/Dden.sum(),
                plugin=plugin,capped=capped,uncapped=uncapped,budget=budget,claim=claim,ratio=ratio,
                prediction_correction=ratio*claim_rows/nrow)
# Different prediction spreads and nonzero claims exercise BOTH correction bounds.
for s in [0,0.1,1,2]:
    exact_mse(ctrl+s*(R-R.mean(0)),Jr*1.5,verify=True)
print('Independent implementation agrees with official kernels, including both correction bounds.')


In [ ]:
# Load generator controls without reading perturbed cells into memory.
import pickle
control_cache=V/'lib/research_mse/h1_generator_controls.pkl'
if control_cache.exists():
    with control_cache.open('rb') as fh: line,train_rows,score_rows=pickle.load(fh)
else:
    h=ad.read_h5ad(V/'lib/h1/H1_full_2025.h5ad',backed='r')
    try:
        line, train_rows, score_rows=gen.h1_controls(h,h.obs['target_gene'].astype(str).to_numpy(),list(genes),2026,'guide_id')
    finally: h.file.close()
    with control_cache.open('wb') as fh: pickle.dump((line,train_rows,score_rows),fh,protocol=5)
settings=json.loads((RUN/'h1_bench_g3/selected_generator.json').read_text())['settings']
cfg=gen.Cfg(argparse.Namespace(**dict(settings,detect_floor=False)))
pdir=RUN/'h1_pred_g3'
pred=np.load(pdir/'raw_predictions.npz')
t300=np.load(pdir/'targets.npy').astype(str); pos={t:i for i,t in enumerate(t300)}; idx=np.array([pos[t] for t in targets])
delta=pred['delta'].astype(float); gcols=pred['gene_cols']; run_set=np.isin(np.arange(G),gcols)
kd=gen.kd_residual(gen.prediction_ontarget(pdir,targets),cfg)
def delta_panel(s=1,beta=1,extension=None):
    dbar=delta.mean(0,keepdims=True)
    D=np.zeros((T,G),np.float32)
    if extension is not None: D[:,~run_set]=np.asarray(extension)[~run_set]
    D[:,gcols]=(beta*dbar+s*(delta-dbar))[idx]
    return D

def bulk_from_calibrated(D):
    P=np.empty((T,G)); factors=np.empty_like(P)
    for i in rows:
        factors[i],share,_=gen.project_factors(line.share,D[i],int(own[i]) if own[i]>=0 else None,kd[i])
        P[i]=np.log1p(50000*share)
    return P,factors

def intended(s=1,beta=1,floor=False,D=None):
    local=copy.copy(cfg); local.detect_floor=floor
    Dc=gen.calibrate(delta_panel(s,beta) if D is None else D,line,targets,local)
    P,F=bulk_from_calibrated(Dc)
    return P,Dc,F
P2,D2,F2=intended()
P1,D1,F1=intended(floor=True)
P7,D7,F7=intended(s=2,floor=True)
print('Control cells:',line.n,'generator;',line.n_reference,'scoring. Model genes:',len(gcols))
for name,P in [('arm1',P1),('arm2',P2),('arm7',P7)]:
    sc=exact_mse(P,verify=True); print(name,'noise-free raw:',sc['raw'],'linear:',scaled_mse(sc['raw']))


In [ ]:
# Reproduce the ORIGINAL A6 terciles; no averaging of tercile ratios.
old=json.loads((SRC/'h1_decomp/results.json').read_text())
size=np.sqrt(np.maximum(((R-R.mean(0))**2).sum(1)-Jr,0))
terc=np.argsort(np.argsort(size))*3//T
num=exact_mse(P2)['capped']; terc_rows=[]
for q,name in enumerate(['weak','mid','strong']):
    sel=terc==q
    terc_rows.append(dict(tercile=name,n=int(sel.sum()),denominator_sum=Dden[sel].sum(),
       denominator_weight=Dden[sel].sum()/Dden.sum(),model_ratio=num[sel].sum()/Dden[sel].sum(),
       baseline_ratio=bl.expr_mse_unbiased_capped.to_numpy()[sel].sum()/Dden[sel].sum()))
terc_table=pd.DataFrame(terc_rows)
display(terc_table)
print('Weighted model:',(terc_table.model_ratio*terc_table.denominator_weight).sum(),
      'baseline:',(terc_table.baseline_ratio*terc_table.denominator_weight).sum())
print('Unweighted tercile model average:',terc_table.model_ratio.mean())
print('A6 sign correction: 0.974 is ABOVE baseline 0.9535, hence negative unclamped skill.')
terc_table.to_csv(OUT/'phase1_terciles.csv',index=False)
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
terc_table.plot.bar(x='tercile',y=['model_ratio','baseline_ratio'],ax=ax[0],rot=0)
terc_table.plot.bar(x='tercile',y='denominator_weight',ax=ax[1],rot=0,legend=False)
ax[0].set_ylabel('Raw MSE within tercile');ax[1].set_ylabel('Weight in full-panel ratio')
fig.tight_layout();fig.savefig(OUT/'phase1_terciles.png',dpi=160);plt.show()


### How to win MSE under this formula

Improve the **sum of absolute squared pseudobulk residuals**, in natural-log bulk space. A large residual in any gene or knockdown remains costly: neither cap limits its damage. Strong-response targets carry most denominator mass, but every target has nearly the same numerator weight (1/18,532 here). Weak targets can have huge individual error/strength ratios while contributing little to the aggregate; optimizing the average of those ratios is the wrong objective.

The shared response is valuable only to the extent its corrected profile reduces that same error sum. The baseline gets to use the target line's real perturbations; learning generic responses from other lines is a genuine prediction problem. Genes outside the model's 5,000 still count. The >5 CPM gate does not protect MSE from errors elsewhere, and composition normalization can move genes assigned zero log2FC.

Generator noise is partly corrected, not automatically removed. Both real and predicted sampling terms enter, but the prediction deduction cannot exceed the real row's jackknife or the panel's own variation budget. Common base-cell errors shared across all knockdowns do not earn across-knockdown budget. Full-control total matching should therefore be tested with these bounds and all six metrics. Do not infer a gain from a two-seed noise reduction alone.

The floor is an especially direct intervention: it turns small predictions into larger shifts regardless of MSE evidence. The current implementation uses a CPM mean/variance z approximation, **not simulated scorer Wilcoxon + BH**; unreachable shifts are assigned 8. Test skipping costly floor changes before broadening model complexity.

Corrections to Part A established from code: raw errors are not capped; both sides have sampling corrections; MSE has no DE CPM gate; common base cells already exist; the current floor still has the unreachable→8 behavior; the baseline is not simply the arithmetic mean of log pseudobulks; and 0.974 is worse, not better, than 0.9535. H1 uses a disjoint control split, not all H1 controls for generation.

Status: phase 1 numerical results above reproduce cached per-target scores and aggregate ratios; independent regeneration of the cells is the first phase 2 check. No submission or existing generator has been modified.


### Model and submission provenance audit

`vcc_2026_2.ipynb`, opening results table and lesson 4, identifies **+0.145 as the improvement from −0.054 to +0.091**. The documented best Model 2 submission is `vcc_2026_3.ipynb`'s `model2_top2000_floor`: `re`, top 2,000 by |Δ|/SE, alpha 1, detection z=5, thinning generator. No separate +0.145 overall is substantiated by the available records.

Model 2 reads the older `lib/sources/<source>/effects_panel.h5ad` tables (`code/vcc_2026_3/vcc2_meta.py:51`), with DerSimonian–Laird per-gene tau² pooled over targets (`:73`) and precision combination (`:100`). It enforces one background vote; `vcc3_ridge.py:66` restricts T-cell evidence to Stim48hr. Model 3's PCA uses uncentered effects and correction ridges; its program-only model is not used here.

`code/transformer/model.py` and `losses.py` match the supplied architecture and loss description, with one nuance: PIE **evidence** is off in dev_nopie, but `gene_features="pie"` still uses PIE gene embeddings, including coverage flags. The saved dev config sets a maximum of 20 epochs and patience 3, not a fixed 12 epochs; the verified best checkpoint is epoch 12, gamma 0.0491102. The recorded prediction checks explicitly show H1 absent from loaded sources, targets, and evidence slots. The effect-bank inputs are `lib/effects_v4`; the panel baseline construction is in `scripts/build_baseline_panel.py`.

Numerical resolution of the tercile puzzle: weak/mid/strong denominator weights are **1.7567% / 11.7402% / 86.5031%**. Their model ratios 1.071720 / 1.114949 / 0.953265 combine to **0.974328**; the baseline combines to **0.953469**. An unweighted tercile average gives 1.046645 and is not the metric.


## Phase 2 — anatomy of the H1 error

Regenerate arms 1, 2, and 7 with the recorded guide-stratified control pool and the original maximum-pool-size rule. Compute the official jackknife on each generated count matrix and apply both caps jointly over the whole 75-target panel. Counts are streamed one target at a time; only pseudobulks and jackknives are cached.

For exact additive mechanism attribution, write the residual as shared + specific + floor + generator. Allocate their cross terms symmetrically: component v receives `sum(v * total_residual)` (its own square plus half of every pairwise interaction). Also report standalone squared magnitudes and actual counterfactual changes; these answer different questions. Separate the per-target mean-over-scored-genes offset orthogonally. Outside-model genes, CPM deciles, and strength terciles are overlapping diagnostic partitions, not extra independent mechanisms. The official scalar noise deductions remain explicit row-level terms; do not invent a gene-specific correction from a scalar cache.


In [ ]:
# Phase 2: regeneration and independent metric reproduction.
max_pool=int(max(400,z['counts'][ri].max()))
pool_cfg=copy.copy(cfg);pool_cfg.n_cells=max_pool
common=gen.pick_cells(line,np.zeros(G),pool_cfg,np.random.default_rng(seed_for(2026,'H1','base_controls')))
print('Common base pool:',max_pool,'; first 400 cells used for every target.',flush=True)
def generated_bulk(name,Dc,seed=2026,mode='original',save_cells=False):
    cache=OUT/f'{name}_seed{seed}_{mode}.npz'
    if cache.exists():
        zz=np.load(cache);return {k:zz[k] for k in zz.files}
    local=copy.copy(cfg);local.seed=seed
    pc=copy.copy(pool_cfg)
    picked=gen.pick_cells(line,np.zeros(G),pc,np.random.default_rng(seed_for(seed,'H1','base_controls')))[:400]
    P=np.empty((T,G));J=np.empty(T);mx=np.empty(T);start=time.time()
    for i,t in enumerate(targets):
        X=gen.generate_one(line,Dc[i],int(own[i]),kd[i],local,pick=picked,line_key='H1',target_key=t)[0]
        counts=np.asarray(X.sum(0)).ravel().astype(float)
        P[i]=np.log1p(50000*counts/counts.sum())
        J[i]=jackknife_correction(X,np.zeros(X.shape[0],np.int32),1,50000,chunk=32)[0]
        mx[i]=np.asarray(X.sum(1)).max()
        if (i+1)%10==0: print(name,i+1,'/',T,'elapsed',round(time.time()-start,1),flush=True)
    result=dict(P=P,J=J,max_cell_counts=mx,picked=picked)
    np.savez_compressed(cache,**result)
    return result
regenerated={}
checks=[]
for name,Dc in [('arm1',D1),('arm2',D2),('arm7',D7)]:
    data=generated_bulk(name,Dc);regenerated[name]=data
    sc=exact_mse(data['P'],data['J'],verify=True);saved=tidy(arm_paths[name]/'results.csv')
    err=np.abs(sc['capped']-saved.expr_mse_unbiased_capped.to_numpy()).max()
    uncerr=np.abs(sc['uncapped']-saved.expr_mse_unbiased.to_numpy()).max()
    checks.append(dict(arm=name,raw_mse=sc['raw'],unclamped=scaled_mse(sc['raw']),
      budget_ratio=sc['ratio'],claim=sc['claim'],budget=sc['budget'],max_row_error=err,
      max_uncapped_row_error=uncerr,mean_Jpred=data['J'].mean(),mean_Jreal=Jr.mean()))
    print(checks[-1],flush=True)
    np.testing.assert_allclose(sc['capped'],saved.expr_mse_unbiased_capped.to_numpy(),atol=3e-8,rtol=3e-5)
checks=pd.DataFrame(checks);checks.to_csv(OUT/'phase2_regeneration_checks.csv',index=False);display(checks)


In [ ]:
# Signed, additive mechanism decomposition; original/raw/capped totals all retained.
P7_no,D7_no,F7_no=intended(s=2,floor=False)
mechanism_rows=[]; slice_rows=[]; gene_tables={}; floors=[]
expr_bin=pd.qcut(pd.Series(line.cpm).rank(method='first'),10,labels=False).to_numpy()
for name,P0,Pi,D0,Df in [('arm2',P2,P2,D2,D2),('arm7',P7_no,P7,D7_no,D7)]:
    Pg=regenerated[name]['P'];Jp=regenerated[name]['J'];sc=exact_mse(Pg,Jp)
    shared=np.broadcast_to(P0.mean(0)-R.mean(0),R.shape).copy()
    specific=(P0-P0.mean(0))-(R-R.mean(0))
    vv={'shared_profile':shared,'knockdown_specific':specific,'floor':Pi-P0,'generator':Pg-Pi}
    residual=Pg-R;np.testing.assert_allclose(sum(vv.values()),residual,atol=1e-14)
    offset=(residual*mask).sum(1)/nrow
    residual_centered=(residual-offset[:,None])*mask
    plugin_sum=float(sc['plugin'].sum());capped_sum=float(sc['capped'].sum())
    for component,v in vv.items():
        allocation=float((v*residual*mask/nrow[:,None]).sum())
        own_square=float((v*v*mask/nrow[:,None]).sum())
        v_offset=(v*mask).sum(1)/nrow
        centered_allocation=float(((v-v_offset[:,None])*residual_centered/nrow[:,None]).sum())
        mechanism_rows.append(dict(arm=name,component=component,signed_contribution=allocation,
           standalone_square=own_square,plugin_share=allocation/plugin_sum,capped_share=allocation/capped_sum,
           raw_metric_contribution=allocation/Dden.sum(),after_offset_contribution=centered_allocation))
    for component,value in [('real_noise_deduction',-float((Jr/nrow).sum())),
                            ('pred_noise_capped_deduction',-float(sc['prediction_correction'].sum()))]:
        mechanism_rows.append(dict(arm=name,component=component,signed_contribution=value,
            standalone_square=np.nan,plugin_share=value/plugin_sum,capped_share=value/capped_sum,
            raw_metric_contribution=value/Dden.sum(),after_offset_contribution=np.nan))
    print(name,'plugin / uncapped / capped:',plugin_sum/Dden.sum(),sc['raw_uncapped'],sc['raw'],
          'offset share of plugin:',float((offset**2).sum()/plugin_sum))
    print(name,'noise-free counterfactual floor raw delta:',exact_mse(Pi)['raw']-exact_mse(P0)['raw'],
          'generator raw delta (including accepted prediction correction):',sc['raw']-exact_mse(Pi)['raw'])
    gene_err=(residual**2*mask/nrow[:,None]).sum(0)
    gene_table=pd.DataFrame({'gene':genes,'control_cpm':line.cpm,'in_model':run_set,
       'plugin_contribution':gene_err,'plugin_share':gene_err/plugin_sum,
       'raw_metric_contribution':gene_err/Dden.sum(),
       'floor_signed_contribution':((Pi-P0)*residual*mask/nrow[:,None]).sum(0),
       'generator_signed_contribution':((Pg-Pi)*residual*mask/nrow[:,None]).sum(0)})
    gene_table=gene_table.sort_values('plugin_contribution',ascending=False);gene_tables[name]=gene_table
    gene_table.head(50).to_csv(OUT/f'phase2_{name}_top50_genes.csv',index=False)
    display(gene_table.head(50))
    for q in range(10):
        use=expr_bin==q
        slice_rows.append(dict(arm=name,partition='expression_decile',group=str(q+1),
             n=int(use.sum()),plugin_sum=gene_err[use].sum(),plugin_share=gene_err[use].sum()/plugin_sum,
             real_correction=np.nan,pred_correction=np.nan,capped_raw=np.nan))
    for label,use in [('in_model',run_set),('outside_model',~run_set)]:
        slice_rows.append(dict(arm=name,partition='model_gene_set',group=label,n=int(use.sum()),
             plugin_sum=gene_err[use].sum(),plugin_share=gene_err[use].sum()/plugin_sum,
             real_correction=np.nan,pred_correction=np.nan,capped_raw=np.nan))
    for q,label in enumerate(['weak','mid','strong']):
        use=terc==q
        slice_rows.append(dict(arm=name,partition='strength_tercile',group=label,n=int(use.sum()),
          plugin_sum=sc['plugin'][use].sum(),plugin_share=sc['plugin'][use].sum()/plugin_sum,
          real_correction=-(Jr/nrow)[use].sum(),pred_correction=-sc['prediction_correction'][use].sum(),
          capped_raw=sc['capped'][use].sum()/Dden[use].sum()))
    raised=(np.abs(Df)>np.abs(D0)+1e-6)&mask
    increments=(np.abs(Df)-np.abs(D0))[raised]
    floor_sq=(Pi-P0)**2*mask/nrow[:,None]
    floor_error_share=(residual**2*raised/nrow[:,None]).sum()/plugin_sum
    print(name,'floor-raised entries',int(raised.sum()),'increment quantiles',
          np.quantile(increments,[0,.25,.5,.75,.9,.99,1]) if len(increments) else [],
          'share of plugin error on raised genes',floor_error_share)
    if name=='arm7':
        for q in range(10):
            use=raised&(expr_bin[None,:]==q)
            floors.append(dict(decile=q+1,raised=int(use.sum()),
              median_delta=float(np.median(np.abs(Df)[use])) if use.any() else np.nan,
              max_delta=float(np.abs(Df)[use].max()) if use.any() else np.nan,
              n_abs_delta_ge3=int((use&(np.abs(Df)>=3)).sum()),
              n_abs_delta_ge7_99=int((use&(np.abs(Df)>=7.99)).sum()),
              plugin_error_share=float((residual**2*use/nrow[:,None]).sum()/plugin_sum),
              standalone_floor_square=float(floor_sq[use].sum())))
        floor_entries=pd.DataFrame({'target':np.repeat(targets,G)[raised.ravel()],
          'gene':np.tile(genes,T)[raised.ravel()], 'cpm':np.broadcast_to(line.cpm,R.shape)[raised],
          'before':D0[raised],'after':Df[raised],'increase':increments})
        floor_entries.nlargest(50,'increase').to_csv(OUT/'phase2_largest_floor_increases.csv',index=False)
        display(floor_entries.nlargest(20,'increase'))
mechanisms=pd.DataFrame(mechanism_rows);slices=pd.DataFrame(slice_rows);floor_table=pd.DataFrame(floors)
mechanisms.to_csv(OUT/'phase2_mechanisms.csv',index=False)
slices.to_csv(OUT/'phase2_partitions.csv',index=False);floor_table.to_csv(OUT/'phase2_floor_deciles.csv',index=False)
display(mechanisms);display(slices);display(floor_table)
for name in ['arm2','arm7']:
    np.testing.assert_allclose(mechanisms.loc[mechanisms.arm==name,'raw_metric_contribution'].sum(),
                              exact_mse(regenerated[name]['P'],regenerated[name]['J'])['raw'],atol=1e-12)
fig,axes=plt.subplots(1,3,figsize=(15,4))
mechanisms.pivot(index='component',columns='arm',values='raw_metric_contribution').plot.barh(ax=axes[0])
slices[slices.partition=='expression_decile'].pivot(index='group',columns='arm',values='plugin_share').reindex([str(i) for i in range(1,11)]).plot.bar(ax=axes[1],rot=0)
axes[1].set_xlabel('Control CPM decile');axes[1].set_ylabel('Share of squared residual')
floor_table.plot.bar(x='decile',y='raised',ax=axes[2],rot=0,legend=False)
axes[2].set_ylabel('Target × gene entries raised by floor')
fig.tight_layout();fig.savefig(OUT/'phase2_anatomy.png',dpi=160);plt.show()


### Phase 2 findings

Independent regenerated-cell numerators reproduce the original scorer per target to **2.4×10⁻¹⁶ or better**, for arms 1, 2 and 7. The phase 1 cached-table reproduction is now also verified from integer counts and recomputed jackknives.

For arm 7, additive signed shares of the uncorrected squared residual are **shared 2.35%, specific 57.11%, floor 33.44%, generator 7.09%**. The scorer then deducts real noise (0.07472 raw units) and accepted prediction noise (0.06267). Removing the floor at fixed s=2 lowers noise-free raw MSE by **0.71024**; generating cells adds back **0.07331** after the scorer's correction. For arm 2, generation adds **0.09057** after correction. Its across-target budget accepts only 69.25% of the already row-capped claim; the full predicted jackknife would deduct 0.14700 raw units, but only 0.04322 is accepted. Noise is therefore materially penalized.

Arm 7's floor raises **341,777 target×gene entries**; the median imposed increase is **0.1341 log2FC**, 99th percentile 0.4378, maximum 1.0049. No retained raised entry reaches |Δ|≥3 or ±8 (the largest resulting |Δ| is 1.0072). The unreachable→8 implementation exists but **does not explain these H1 arms**. Broad, modest changes in expressed genes cause the damage; caps 2 and 3 cannot help this dataset.

Genes outside the model account for **40.55%** of arm 7's squared residual and **65.35%** of arm 2's. The per-target global-offset component is only **0.739% / 0.239%** respectively; E4 has a small oracle ceiling. The top-gene and partition tables above retain the scalar correction terms separately, because a scalar cache cannot uniquely attribute them to genes.

Additional provenance correction: the transformer's `basal_table` prefers `*_baseline_v2.csv` when present. Those features normalize to the **6,013 shared genes** (`scripts/build_baseline_v2.py`), with a separate detection-missing flag; panel-only baselines are a fallback. This differs from Part A's simpler baseline-panel description and is preserved rather than silently changing the model inputs.
